In [10]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

base_path = Path("../../..")  # from src/mcq/preprocessing/
data_dir = base_path / "data" / "processed" / "mcq"

# Unified dataset: all diseases in one CSV with one-hot cols (depression, anxiety, ocd, adhd)
all_diseases_path = data_dir / "all_diseases_dataset.csv"

In [11]:
# Load unified MCQ dataset and inspect basic structure

df = pd.read_csv(all_diseases_path)

print("=== All Diseases dataset ===")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

=== All Diseases dataset ===
Shape: (8000, 20)
Columns: ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'depression', 'anxiety', 'ocd', 'adhd', 'label']


,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10,q11,q12,q13,q14,q15,depression,anxiety,ocd,adhd,label
0,2,3,3,3,3,2,2,3,2,2,2,3,2,2,2,1,0,0,0,1
1,2,0,1,0,1,0,1,1,1,0,0,1,1,1,0,1,0,0,0,0
2,2,2,2,2,2,2,3,2,2,3,2,2,3,2,2,1,0,0,0,1
3,0,1,1,2,0,0,0,1,1,1,0,2,0,1,0,1,0,0,0,0
4,3,3,3,1,3,2,2,3,3,2,2,3,1,3,2,1,0,0,0,1


In [12]:
# Check missing values (NaN) and -1 placeholders

feature_cols = [c for c in df.columns if c != "label"]
print("NaN per column:\n", df.isna().sum())
print("\n-1 per feature (treated as missing):\n", (df[feature_cols] == -1).sum())
print("Total rows:", len(df))

NaN per column:
 q1            0
q2            0
q3            0
q4            0
q5            0
q6            0
q7            0
q8            0
q9            0
q10           0
q11           0
q12           0
q13           0
q14           0
q15           0
depression    0
anxiety       0
ocd           0
adhd          0
label         0
dtype: int64

-1 per feature (treated as missing):
 q1            0
q2            0
q3            0
q4            0
q5            0
q6            0
q7            0
q8            0
q9            0
q10           0
q11           0
q12           0
q13           0
q14           0
q15           0
depression    0
anxiety       0
ocd           0
adhd          0
dtype: int64
Total rows: 8000


In [13]:
# Handle missing values: convert -1 to NaN and impute with column medians

import numpy as np

feature_cols = [c for c in df.columns if c != "label"]
df_clean = df.copy()

# Replace -1 with NaN in feature columns
df_clean[feature_cols] = df_clean[feature_cols].replace(-1, np.nan)

# Impute NaN with column median for each feature
medians = df_clean[feature_cols].median()
df_clean[feature_cols] = df_clean[feature_cols].fillna(medians)

print("Any remaining NaN?", df_clean.isna().any().any())

Any remaining NaN? False


In [14]:
# Split into features (X) and labels (y)
# Features: q1..q15 + depression, anxiety, ocd, adhd (one-hot)

X = df_clean.drop(columns=["label"])
y = df_clean["label"]
print(f"X shape = {X.shape}, y shape = {y.shape}")

X shape = (8000, 19), y shape = (8000,)


In [15]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"X_train={X_train.shape}, X_test={X_test.shape}, y_train={y_train.shape}, y_test={y_test.shape}")

X_train=(6400, 19), X_test=(1600, 19), y_train=(6400,), y_test=(1600,)


In [16]:
# Define models for MCQ classification (unified dataset)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

estimators = {
    "logreg": lambda: LogisticRegression(max_iter=1000, n_jobs=-1),
    "rf": lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "xgb": lambda: XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42,
    ),
}

In [17]:
# Train logistic regression, XGBoost, and Random Forest with 5-fold CV

cv_results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for algo_name, make_est in estimators.items():
    model = make_est()
    acc_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    f1_scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
    cv_results[algo_name] = {
        "accuracy_mean": acc_scores.mean(),
        "accuracy_std": acc_scores.std(),
        "f1_macro_mean": f1_scores.mean(),
        "f1_macro_std": f1_scores.std(),
    }
    print(
        f"{algo_name}: acc={acc_scores.mean():.3f}±{acc_scores.std():.3f}, "
        f"f1_macro={f1_scores.mean():.3f}±{f1_scores.std():.3f}"
    )

logreg: acc=1.000±0.000, f1_macro=1.000±0.000
rf: acc=1.000±0.000, f1_macro=1.000±0.000
xgb: acc=1.000±0.000, f1_macro=1.000±0.000


In [18]:
# Fit final models on training data and evaluate on held-out test set

fitted_models = {}

for algo_name, make_est in estimators.items():
    model = make_est()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    print(f"\n===== Model: {algo_name} =====")
    print(f"Accuracy: {acc:.3f}, Macro-F1: {f1:.3f}")
    print("Classification report:\n", classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

    fitted_models[algo_name] = model

e:\Interview-AI-Detection\interview-ai-detection\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



===== Model: logreg =====
Accuracy: 1.000, Macro-F1: 1.000
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       800
           1       1.00      1.00      1.00       800

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600

Confusion matrix:
 [[800   0]
 [  0 800]]

===== Model: rf =====
Accuracy: 1.000, Macro-F1: 1.000
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       800
           1       1.00      1.00      1.00       800

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600

Confusion matrix:
 [[800   0]
 [  0 800]]

===== Model: xgb =====
Accuracy: 1.000, Macro-F1: 1.000
Classification report:
               pre

In [19]:
# Compute and display feature importance (which questions matter most)

import numpy as np

feature_names = list(X.columns)
feature_importances = {}

for algo_name, model in fitted_models.items():
    if algo_name == "logreg":
        coefs = np.abs(model.coef_)
        importances = coefs.mean(axis=0) if coefs.ndim == 2 else coefs
    else:
        importances = getattr(model, "feature_importances_", None)

    if importances is None:
        print(f"{algo_name}: no feature_importances_ available")
        continue

    importance_pairs = sorted(
        zip(feature_names, importances), key=lambda x: x[1], reverse=True
    )
    feature_importances[algo_name] = importance_pairs

    print(f"\n===== Top features for {algo_name} =====")
    for fname, score in importance_pairs[:15]:
        print(f"  {fname}: {score:.4f}")


===== Top features for logreg =====
  q4: 1.0684
  q5: 1.0627
  q1: 1.0587
  q12: 1.0427
  q3: 1.0287
  q2: 0.9990
  q13: 0.9890
  q8: 0.9375
  q6: 0.9274
  q7: 0.9212
  q11: 0.8979
  q14: 0.8941
  q15: 0.8903
  q9: 0.8825
  q10: 0.8641

===== Top features for rf =====
  q1: 0.1304
  q12: 0.1073
  q9: 0.0911
  q15: 0.0871
  q5: 0.0860
  q14: 0.0817
  q2: 0.0666
  q11: 0.0509
  q3: 0.0479
  q7: 0.0477
  q4: 0.0461
  q6: 0.0446
  q13: 0.0384
  q10: 0.0375
  q8: 0.0364

===== Top features for xgb =====
  q1: 0.1541
  q12: 0.1113
  q15: 0.0981
  q14: 0.0867
  q9: 0.0834
  q2: 0.0683
  q5: 0.0585
  q4: 0.0552
  q6: 0.0545
  q7: 0.0484
  q11: 0.0464
  q10: 0.0393
  q3: 0.0359
  q8: 0.0320
  q13: 0.0279


In [21]:
# Persist trained models to disk under models/mcq_model

import joblib

models_dir = base_path / "models" / "mcq_model"
models_dir.mkdir(parents=True, exist_ok=True)

for algo_name, model in fitted_models.items():
    model_path = models_dir / f"mcq_{algo_name}.joblib"
    joblib.dump(model, model_path)
    print(f"Saved {algo_name} model to {model_path}")

Saved logreg model to ..\..\..\models\mcq_model\mcq_logreg.joblib
Saved rf model to ..\..\..\models\mcq_model\mcq_rf.joblib
Saved xgb model to ..\..\..\models\mcq_model\mcq_xgb.joblib
